# Police Force Crime Data Pipeline
## Phase 1 - Data Engineering

**Assignment:** Python Project - Crime Data Pipeline & Reporting Dataset  
**Reporting grain:** Police Force x Month x Crime Category  
**Forces covered:** Metropolitan Police Service, West Midlands Police, West Yorkshire Police, Devon & Cornwall Police  
**Time period:** April 2023 -> March 2026 (36 months)

---

This notebook takes raw UK Police street-level crime data, cleans and enriches it using three external datasets,combining it into a single aggregated CSV exported to the 'outputs/' folder.

## Section 0 - Setup & Configuration

Before working with the data, all the key settings are centralised in one place. 

Such as: 
- file paths
- force names
- population estimates
- keyword mappings (file detection)

Ensuring that the pipeline stays flexible and updatable.

In [2]:
# standard library
import os
import re
import warnings

# data manipulation 
import pandas as pd
import numpy as np

# utilities displayed
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("all imports loaded successfully!!")

all imports loaded successfully!!


In [3]:
# path configuration 
from pathlib import Path

# notebook sits at the project root (same level as crime-data/, extra-data/,
# and outputs/). Path(__file__) is not available in jupyter
# globals()['_dh'][0] which always resolves to the folder containing the .ipynb file no matter where Jupyter was launched from.

BASE_DIR  = Path(globals()['_dh'][0])

CRIME_DIR  = BASE_DIR / "crime-data"
EXTRA_DIR  = BASE_DIR / "extra-data"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# enrichment file paths 
POPULATION_FILE = EXTRA_DIR / "population-23-25.csv"
IMD_FILE        = EXTRA_DIR / "Index_of_Multiple_Deprivation.xlsx"
SEVERITY_FILE   = EXTRA_DIR / "severity-score.xls"
OUTPUT_FILE     = OUTPUT_DIR / "crime_reporting_dataset.csv"

print(f"BASE_DIR   : {BASE_DIR}")
print(f"CRIME_DIR  : {CRIME_DIR}")
print(f"EXTRA_DIR  : {EXTRA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# sanity check - fails if the folder structure is wrong
assert CRIME_DIR.exists(),  f"crime-data folder not found at: {CRIME_DIR}"
assert EXTRA_DIR.exists(),  f"extra-data folder not found at: {EXTRA_DIR}"
print("folder structure verified!!!")

BASE_DIR   : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning
CRIME_DIR  : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\crime-data
EXTRA_DIR  : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\extra-data
OUTPUT_DIR : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\outputs
folder structure verified!!!


In [4]:
# force name configuration 

# canonical names used throughout the pipeline.
FORCES = [
    "Metropolitan Police Service",
    "West Midlands Police",
    "West Yorkshire Police",
    "Devon & Cornwall Police",
]

# maps filename keywords - canonical force name.
FORCE_FILE_KEYWORDS = {
    "metropolitan":       "Metropolitan Police Service",
    "west-midlands":      "West Midlands Police",
    "west-yorkshire":     "West Yorkshire Police",
    "devon-and-cornwall": "Devon & Cornwall Police",
}

# raw "falls within" / "reported by" variants - canonical names.
FORCE_NAME_MAP = {
    "Metropolitan Police Service":     "Metropolitan Police Service",
    "West Midlands Police":            "West Midlands Police",
    "West Yorkshire Police":           "West Yorkshire Police",
    "Devon & Cornwall Constabulary":   "Devon & Cornwall Police",
    "Devon and Cornwall Constabulary": "Devon & Cornwall Police",
    "Devon and Cornwall Police":       "Devon & Cornwall Police",
    "Devon & Cornwall Police":         "Devon & Cornwall Police",
}

#  LAD code (2021) - PFA canonical name 
# FORCE_POPULATION (hardcoded dictionary) has been removed and replaced by a
# data-driven LAD - PFA aggregation in Section 4b. This mapping is the bridge:
# 2021 ONS LAD codes - canonical PFA name. Only the four PFAs in scope are
# included; all other LADs in the CSV are filtered out.
#
LAD_TO_PFA = {
    # Metropolitan Police Service (33 London Boroughs)
    "E09000001": "Metropolitan Police Service",  # City of London
    "E09000002": "Metropolitan Police Service",  # Barking and Dagenham
    "E09000003": "Metropolitan Police Service",  # Barnet
    "E09000004": "Metropolitan Police Service",  # Bexley
    "E09000005": "Metropolitan Police Service",  # Brent
    "E09000006": "Metropolitan Police Service",  # Bromley
    "E09000007": "Metropolitan Police Service",  # Camden
    "E09000008": "Metropolitan Police Service",  # Croydon
    "E09000009": "Metropolitan Police Service",  # Ealing
    "E09000010": "Metropolitan Police Service",  # Enfield
    "E09000011": "Metropolitan Police Service",  # Greenwich
    "E09000012": "Metropolitan Police Service",  # Hackney
    "E09000013": "Metropolitan Police Service",  # Hammersmith and Fulham
    "E09000014": "Metropolitan Police Service",  # Haringey
    "E09000015": "Metropolitan Police Service",  # Harrow
    "E09000016": "Metropolitan Police Service",  # Havering
    "E09000017": "Metropolitan Police Service",  # Hillingdon
    "E09000018": "Metropolitan Police Service",  # Hounslow
    "E09000019": "Metropolitan Police Service",  # Islington
    "E09000020": "Metropolitan Police Service",  # Kensington and Chelsea
    "E09000021": "Metropolitan Police Service",  # Kingston upon Thames
    "E09000022": "Metropolitan Police Service",  # Lambeth
    "E09000023": "Metropolitan Police Service",  # Lewisham
    "E09000024": "Metropolitan Police Service",  # Merton
    "E09000025": "Metropolitan Police Service",  # Newham
    "E09000026": "Metropolitan Police Service",  # Redbridge
    "E09000027": "Metropolitan Police Service",  # Richmond upon Thames
    "E09000028": "Metropolitan Police Service",  # Southwark
    "E09000029": "Metropolitan Police Service",  # Sutton
    "E09000030": "Metropolitan Police Service",  # Tower Hamlets
    "E09000031": "Metropolitan Police Service",  # Waltham Forest
    "E09000032": "Metropolitan Police Service",  # Wandsworth
    "E09000033": "Metropolitan Police Service",  # Westminster

    # West Midlands Police (7 metropolitan districts)
    "E08000025": "West Midlands Police",  # Birmingham
    "E08000026": "West Midlands Police",  # Coventry
    "E08000027": "West Midlands Police",  # Dudley
    "E08000028": "West Midlands Police",  # Sandwell
    "E08000029": "West Midlands Police",  # Solihull
    "E08000030": "West Midlands Police",  # Walsall
    "E08000031": "West Midlands Police",  # Wolverhampton

    # West Yorkshire Police (5 metropolitan districts)
    "E08000032": "West Yorkshire Police",  # Bradford
    "E08000033": "West Yorkshire Police",  # Calderdale
    "E08000034": "West Yorkshire Police",  # Kirklees
    "E08000035": "West Yorkshire Police",  # Leeds
    "E08000036": "West Yorkshire Police",  # Wakefield

    # Devon & Cornwall Police (Cornwall + Devon districts + Plymouth + Torbay)
    "E06000052": "Devon & Cornwall Police",  # Cornwall
    "E06000053": "Devon & Cornwall Police",  # Isles of Scilly
    "E10000008": "Devon & Cornwall Police",  # Devon (county)
    "E07000040": "Devon & Cornwall Police",  # East Devon
    "E07000041": "Devon & Cornwall Police",  # Exeter
    "E07000042": "Devon & Cornwall Police",  # Mid Devon
    "E07000043": "Devon & Cornwall Police",  # North Devon
    "E07000044": "Devon & Cornwall Police",  # South Hams
    "E07000045": "Devon & Cornwall Police",  # Teignbridge
    "E07000046": "Devon & Cornwall Police",  # Torridge
    "E07000047": "Devon & Cornwall Police",  # West Devon
    "E06000026": "Devon & Cornwall Police",  # Plymouth
    "E06000027": "Devon & Cornwall Police",  # Torbay
}

# ai disclosure - LAD code to PFA canonical name was derived with help from ai 

# severity score name mapping
SEVERITY_NAME_MAP = {
    "Metropolitan Police": "Metropolitan Police Service",
    "West Midlands":       "West Midlands Police",
    "West Yorkshire":      "West Yorkshire Police",
    "Devon and Cornwall":  "Devon & Cornwall Police",
}

# time period
START_MONTH = "2023-04"
END_MONTH   = "2026-03"
YEARS       = [2023, 2024, 2025, 2026]

print("Configuration loaded.")
print(f"Forces: {FORCES}")
print(f"LAD_TO_PFA: {len(LAD_TO_PFA)} LAD codes mapped across {len(FORCES)} PFAs")
print(f"Time window: {START_MONTH} → {END_MONTH}")


Configuration loaded.
Forces: ['Metropolitan Police Service', 'West Midlands Police', 'West Yorkshire Police', 'Devon & Cornwall Police']
LAD_TO_PFA: 58 LAD codes mapped across 4 PFAs
Time window: 2023-04 → 2026-03


---
## Section 1 - Ingestion Layer

Before loading the files, the folder structure will first be verified to ensure the expected monthly subdirectories are present. Reads only the four required force files for each month before concatenating them into a single raw DataFrame at the end.

Loading the entire dataset in a single operation would likely exceed the memory capacity of a standard machine, as the raw CSV files total approximately 1.3 GB across 36 months and four police forces. Therefore, processing the data in monthly batches allows progress to be monitored more effectively.

In [5]:
# folder structure validation
import re

monthly_folders = sorted([
    f.name for f in CRIME_DIR.iterdir()
    if f.is_dir() and re.match(r"\d{4}-\d{2}", f.name)
])

print(f"monthly folders found: {len(monthly_folders)}")
print(f"first: {monthly_folders[0]}  |  last: {monthly_folders[-1]}")

# check that we have the months wanted
expected_months = pd.period_range(start=START_MONTH, end=END_MONTH, freq="M").astype(str).tolist()
missing = set(expected_months) - set(monthly_folders)
extra   = set(monthly_folders) - set(expected_months)

if missing:
    print(f"missing folders: {sorted(missing)}")
else:
    print("all 36 expected monthly folders are present!!")

if extra:
    print(f"extra folders found (will be skipped): {sorted(extra)}")

monthly folders found: 36
first: 2023-04  |  last: 2026-03
all 36 expected monthly folders are present!!


In [6]:
# loop batch ingestion

chunks = []  # accumulate DataFrames here; concatenate once at the end

for month_folder in monthly_folders:
    folder_path = CRIME_DIR / month_folder
    files_in_folder = [f.name for f in folder_path.iterdir() if f.is_file()]
    month_chunks = []

    for keyword, force_name in FORCE_FILE_KEYWORDS.items():
        # Find the file whose name contains this force's keyword
        matches = [f for f in files_in_folder if keyword in f and f.endswith(".csv")]

        if not matches:
            print(f"  [{month_folder}] no file found for keyword '{keyword}'")
            continue

        filepath = folder_path / matches[0]

        # reads the CSV — dtype=str keeps everything as strings on load;
        # cast types in the cleaning stage
        df_chunk = pd.read_csv(filepath, dtype=str, low_memory=False)
        month_chunks.append(df_chunk)

    if month_chunks:
        month_df = pd.concat(month_chunks, ignore_index=True)
        chunks.append(month_df)
        print(f"  [{month_folder}] loaded {len(month_df):>8,} rows across {len(month_chunks)} force files")

print(f"all monthly batches processed. Concatenating into raw DataFrame...")
df_raw = pd.concat(chunks, ignore_index=True)
print(f"done!!!")

  [2023-04] loaded  154,835 rows across 4 force files
  [2023-05] loaded  164,785 rows across 4 force files
  [2023-06] loaded  174,208 rows across 4 force files
  [2023-07] loaded  166,604 rows across 4 force files
  [2023-08] loaded  160,283 rows across 4 force files
  [2023-09] loaded  160,805 rows across 4 force files
  [2023-10] loaded  168,389 rows across 4 force files
  [2023-11] loaded  157,010 rows across 4 force files
  [2023-12] loaded  155,906 rows across 4 force files
  [2024-01] loaded  153,592 rows across 4 force files
  [2024-02] loaded  150,436 rows across 4 force files
  [2024-03] loaded  158,105 rows across 4 force files
  [2024-04] loaded  156,655 rows across 4 force files
  [2024-05] loaded  170,061 rows across 4 force files
  [2024-06] loaded  167,186 rows across 4 force files
  [2024-07] loaded  174,052 rows across 4 force files
  [2024-08] loaded  168,178 rows across 4 force files
  [2024-09] loaded  160,218 rows across 4 force files
  [2024-10] loaded  169,155 

In [7]:
# summary raw dataset 
print(f"shape         : {df_raw.shape}")
print(f"rows          : {df_raw.shape[0]:,}")
print(f"columns       : {df_raw.shape[1]}")
print(f"memory usage  : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()
print("Columns:", df_raw.columns.tolist())
print()
print("Sample (5 rows):")
display(df_raw.head())

shape         : (5757613, 12)
rows          : 5,757,613
columns       : 12
memory usage  : 4539.1 MB

Columns: ['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']

Sample (5 rows):


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,7c85d5925620a55fd890ea37dd748a34d2c77773f5d740...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.254186,50.834677,On or near Kingsland Close,E01031372,Adur 004E,Vehicle crime,Investigation complete; no suspect identified,NaN
1,43ccf73d9d380e4fcd128d7384d49ccf8d5ed54887a940...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.559343,50.852819,On or near School Lane,E01031392,Arun 001C,Violence and sexual offences,Investigation complete; no suspect identified,NaN
2,dd453aa29e0058e3fd98c507bf9429b10d654b653571f7...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.405658,50.865992,On or near Southview Road,E01031425,Arun 002C,Violence and sexual offences,Investigation complete; no suspect identified,NaN
3,97181c7698bd151ab9f2940a048b4b26a921192fa9656c...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.481480,50.821606,On or near Willowwood Close,E01031389,Arun 005B,Violence and sexual offences,Status update unavailable,NaN
4,bfc7e3371879ba9e2dc2240a146c797662d01a3ecb4e75...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.543948,50.816593,On or near Malthouse Passage,E01031469,Arun 009F,Violence and sexual offences,Status update unavailable,NaN


---
## Section 2 — Cleaning & Validation Layer

Raw data from Police.uk known issues to address:

- The `Context` column is always null, it can be dropped.
- `Crime ID` is not a primary key. Some record types don't receive a Crime ID at the point of recording. Thus Null Crime IDs are expected and are **not** a data quality problem - rows wont be dropped.
- `LSOA code` is null for records where the exact location cannot be disclosed. These rows are retained, but proportion will be flagged, as they won't match the IMD enrichment join.
- Force name variants need standardising to a single canonical label before we can aggregate correctly.

(row counts will be tracked before and after every transformation)

In [8]:
# row count - before cleaning
rows_before_cleaning = len(df_raw)
print(f"rows BEFORE cleaning: {rows_before_cleaning:,}")

df = df_raw.copy()  

rows BEFORE cleaning: 5,757,613


In [9]:
# 'Context' column dropped
# column is documented as always null — confirmed empirically.

context_non_null = df["Context"].notna().sum()
print(f"Non-null values in 'Context': {context_non_null}")
df.drop(columns=["Context"], inplace=True)
print("'Context' column dropped.")

Non-null values in 'Context': 0
'Context' column dropped.


In [10]:
# Crime ID: null analysis
# Crime ID is a record-level hash, and its also NOT a reliable unique identifier.
# absence is normal for certain record types

null_crime_id = df["Crime ID"].isna().sum()
pct_null_crime_id = null_crime_id / len(df) * 100
print(f"Null Crime IDs : {null_crime_id:,} ({pct_null_crime_id:.1f}% of all records)")
print("retained as-is. Null Crime ID is expected and not treated as an error.")
print("Crime ID is NOT used as a primary key anywhere in this pipeline.")

Null Crime IDs : 923,555 (16.0% of all records)
retained as-is. Null Crime ID is expected and not treated as an error.
Crime ID is NOT used as a primary key anywhere in this pipeline.


In [11]:
# LSOA code: null analysis and flagging 
# records without an LSOA code typically have a location suppressed for 'privacy'
# retained but track proportion, because they will produce NaN values after the IMD join.
null_lsoa = df["LSOA code"].isna().sum()
pct_null_lsoa = null_lsoa / len(df) * 100
print(f"Null LSOA codes  : {null_lsoa:,} ({pct_null_lsoa:.1f}%)")

# boolean flag column added
df["lsoa_missing_flag"] = df["LSOA code"].isna()
print(f"'lsoa_missing_flag' column added.")

Null LSOA codes  : 45,375 (0.8%)
'lsoa_missing_flag' column added.


In [12]:
# crime type standardised

df["Crime type"] = df["Crime type"].str.strip().str.title()

unique_crime_types = sorted(df["Crime type"].dropna().unique().tolist())
print(f"UNIQUE crime categories ({len(unique_crime_types)}):")
for ct in unique_crime_types:
    print(f"  · {ct}")

UNIQUE crime categories (14):
  · Anti-Social Behaviour
  · Bicycle Theft
  · Burglary
  · Criminal Damage And Arson
  · Drugs
  · Other Crime
  · Other Theft
  · Possession Of Weapons
  · Public Order
  · Robbery
  · Shoplifting
  · Theft From The Person
  · Vehicle Crime
  · Violence And Sexual Offences


In [13]:
# force name standardised
# 'Falls within' column contains raw force names from each CSV.
# Devon & Cornwall appears under multiple variants
# ('Constabulary' vs 'Police'). map everything to canonical names.
raw_variants = df["Falls within"].unique().tolist()
print(f"Raw 'Falls within' variants found: {raw_variants}")
print()

df["Falls within"] = df["Falls within"].map(FORCE_NAME_MAP)

# Sanity check: are there any unmapped values?
unmapped = df["Falls within"].isna().sum()
if unmapped > 0:
    print(f" {unmapped:,} rows have unmapped force names - review FORCE_NAME_MAP.")
else:
    print("all force name variants successfully mapped to canonical names!")
    print(f" Canonical values: {df['Falls within'].unique().tolist()}")

Raw 'Falls within' variants found: ['Metropolitan Police Service', 'West Midlands Police', 'West Yorkshire Police', 'Devon & Cornwall Police']

all force name variants successfully mapped to canonical names!
 Canonical values: ['Metropolitan Police Service', 'West Midlands Police', 'West Yorkshire Police', 'Devon & Cornwall Police']


In [14]:
# checking duplicates
# functional duplicate - two rows sharing the same:
#   Month + Falls within + Crime type + Longitude + Latitude + Crime ID
# Crime ID condition is applied only where Crime ID is NOT null,
# rows with null Crime IDs are not checked against each other 

df_with_id = df[df["Crime ID"].notna()].copy()

dupe_cols = ["Month", "Falls within", "Crime type", "Longitude", "Latitude", "Crime ID"]
duplicates_mask = df_with_id.duplicated(subset=dupe_cols, keep="first")
n_dupes = duplicates_mask.sum()

print(f"records with non-null Crime ID : {len(df_with_id):,}")
print(f"functional duplicates identified: {n_dupes:,}")

if n_dupes > 0:
    print(f"removing {n_dupes:,} duplicates...")
    dupe_indices = df_with_id[duplicates_mask].index
    df = df.drop(index=dupe_indices).reset_index(drop=True)
    print(f"duplicates removed.")
else:
    print("no functional duplicates found.")

records with non-null Crime ID : 4,834,058
functional duplicates identified: 29,534
removing 29,534 duplicates...
duplicates removed.


In [15]:
# Row count - after cleaning 
rows_after_cleaning = len(df)
rows_removed = rows_before_cleaning - rows_after_cleaning
print(f"rows BEFORE cleaning : {rows_before_cleaning:,}")
print(f"rows AFTER  cleaning : {rows_after_cleaning:,}")
print(f"rows removed         : {rows_removed:,} ({rows_removed/rows_before_cleaning*100:.2f}%)")

rows BEFORE cleaning : 5,757,613
rows AFTER  cleaning : 5,728,079
rows removed         : 29,534 (0.51%)


---
## Section 3 — Feature Engineering

With the data cleaned, its possible to derrive the time attributes needed for aggregation and BI reporting. Including:
- `datetime` representation of the month
- Extracted `year` and `month_num` for time-series filtering in Power BI
- A `quarter` label for seasonal analysis
- A clean `force_name` column 

In [16]:
# month to datetime 
# source format:'YYYY-MM' 

df["month_period"] = df["Month"]  # original string preserved for reference

df["month_dt"] = pd.to_datetime(df["Month"], format="%Y-%m")

print("month to datetime updated successfully.")
print(f"Min month : {df['month_dt'].min().strftime('%Y-%m')}")
print(f"Max month : {df['month_dt'].max().strftime('%Y-%m')}")

month to datetime updated successfully.
Min month : 2023-04
Max month : 2026-03


In [17]:
# year, month_num, quarter - derrived
df["year"]      = df["month_dt"].dt.year
df["month_num"] = df["month_dt"].dt.month

# quarter label (e.g. '2023-Q2') - seasonal comparisons
df["quarter"] = df["year"].astype(str) + "-Q" + df["month_dt"].dt.quarter.astype(str)

# canonical force name column 
df["force_name"] = df["Falls within"]

# rename to snake_case 

df.rename(columns={"Crime type": "crime_type"}, inplace=True)

print("derived columns added:")
sample_cols = ["month_period", "month_dt", "year", "month_num", "quarter", "force_name", "crime_type"]
display(df[sample_cols].drop_duplicates().sort_values("month_dt").head(10))

derived columns added:


,month_period,month_dt,year,month_num,quarter,force_name,crime_type
0,2023-04,2023-04-01,2023,4,2023-Q2,Metropolitan Police Service,Vehicle Crime
116234,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Criminal Damage And Arson
116235,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Shoplifting
116236,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Violence And Sexual Offences
116243,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Drugs
116254,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Other Theft
116257,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Theft From The Person
116258,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Vehicle Crime
116269,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Other Crime
116271,2023-04,2023-04-01,2023,4,2023-Q2,West Yorkshire Police,Burglary


---
## Section 4 - Enrichment Joins

### 4a - Index of Multiple Deprivation (LSOA-level join, before aggregation)

**Why this dataset?**  
The IMD 2019 provides a nationally standardised deprivation ranking for every LSOA in England. By joining it at the record level (before aggregation), a mean deprivation decile can be computed for each force × month × crime-type group


In [18]:
# load IMD 2019
imd_raw = pd.read_excel(
    IMD_FILE,
    sheet_name="IMD2019",    # ignore the 'Notes' sheet
    dtype=str
)

print(f"IMD raw shape: {imd_raw.shape}")
print("Columns:", imd_raw.columns.tolist())

IMD raw shape: (32844, 6)
Columns: ['LSOA code (2011)', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Rank', 'Index of Multiple Deprivation (IMD) Decile']


In [19]:
# select and rename the relevant IMD columns
imd = imd_raw[[
    "LSOA code (2011)",
    "Index of Multiple Deprivation (IMD) Rank",
    "Index of Multiple Deprivation (IMD) Decile"
]].copy()

imd.rename(columns={
    "LSOA code (2011)":                             "lsoa_code",
    "Index of Multiple Deprivation (IMD) Rank":     "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile":   "imd_decile",
}, inplace=True)

# cast numeric columns (came in as strings)
imd["imd_rank"]   = pd.to_numeric(imd["imd_rank"],   errors="coerce")
imd["imd_decile"] = pd.to_numeric(imd["imd_decile"], errors="coerce")

print(f"IMD records loaded: {len(imd):,} LSOAs")
print(f"IMD decile range  : {imd['imd_decile'].min():.0f}–{imd['imd_decile'].max():.0f}")
print()
display(imd.head())

IMD records loaded: 32,844 LSOAs
IMD decile range  : 1–10



,lsoa_code,imd_rank,imd_decile
0,E01000001,29199,9
1,E01000002,30379,10
2,E01000003,14915,5
3,E01000005,8678,3
4,E01000006,14486,5


In [20]:
# left join: crime records - IMD (on LSOA code) 
# LEFT join - records without an LSOA code (or with an LSOA not present in the 2019 IMD

df = df.merge(
    imd,
    left_on="LSOA code",
    right_on="lsoa_code",
    how="left"
)

# match rate analysis
matched      = df["imd_decile"].notna().sum()
unmatched    = df["imd_decile"].isna().sum()
pct_matched  = matched   / len(df) * 100
pct_unmatched = unmatched / len(df) * 100

print(f"total records AFTER join : {len(df):,}")
print(f"IMD matched              : {matched:,}   ({pct_matched:.1f}%)")
print(f"IMD unmatched (NaN)      : {unmatched:,}  ({pct_unmatched:.1f}%)")
print()

total records AFTER join : 5,728,079
IMD matched              : 5,190,375   (90.6%)
IMD unmatched (NaN)      : 537,704  (9.4%)



### 4b — Population Estimates (LAD - PFA aggregation)

**Why this dataset?**  
Raw crime counts are difficult to compare fairly between forces with very different populations. The Metropolitan Police serves ~9 million people; Devon & Cornwall covers ~1.7 million. Per-capita normalisation (`crimes per 1,000 residents`) thus making cross-force comparison meaningful.

**Grain:** Force x Year. The same annual figure applies to all 12 months within each year.

In [21]:
# population file inspection
print("- Population Enrichment -")

if not POPULATION_FILE.exists():
    raise FileNotFoundError(
        f"Population file not found: {POPULATION_FILE}\n"
        "Ensure the ONS LAD-level mid-year estimates CSV is in data/extra-data/"
    )

# snippet of raw structure to confirm column names before full load
print("\n- raw file snippet (first 5 rows) -")
df_pop_preview = pd.read_csv(POPULATION_FILE, nrows=5)
display(df_pop_preview)
print("detected columns:", df_pop_preview.columns.tolist())

# Full load
df_pop_raw = pd.read_csv(POPULATION_FILE)
print(f"\n loaded full file — {df_pop_raw.shape[0]:,} rows × {df_pop_raw.shape[1]} columns")

- Population Enrichment -

- raw file snippet (first 5 rows) -


,geography code,geography name,year,All Ages
0,E06000026,Plymouth,2023,272067
1,E06000026,Plymouth,2024,272067
2,E06000026,Plymouth,2025,272067
3,E06000027,Torbay,2023,140126
4,E06000027,Torbay,2024,140126


detected columns: ['geography code', 'geography name', 'year', 'All Ages']

 loaded full file — 174 rows × 4 columns


In [22]:
# detect column names dynamically

cols = df_pop_raw.columns.str.strip().tolist()

# LAD code column
LAD_CODE_CANDIDATES = [
    "geography code", "Geography code", "Code", "LAD code", "lad_code",
    "area_code", "Area code", "LADCD", "ladcd",
]
lad_col = next((c for c in LAD_CODE_CANDIDATES if c in cols), None)
if lad_col is None:
    raise KeyError(
        f"could not identify LAD code column.\nAvailable: {cols}\n"
        f"searched for: {LAD_CODE_CANDIDATES}"
    )

# Year column
YEAR_CANDIDATES = ["year", "Year", "mid_year", "Mid Year", "period", "Period"]
year_col = next((c for c in YEAR_CANDIDATES if c in cols), None)
if year_col is None:
    raise KeyError(
        f"could not identify year column.\nAvailable: {cols}\n"
        f"searched for: {YEAR_CANDIDATES}"
    )

# All-ages total population column
POP_CANDIDATES = [
    "All Ages", "all_ages", "total", "Total", "population", "Population",
    "all ages", "All ages", "persons", "Persons",
]
pop_col = next((c for c in POP_CANDIDATES if c in cols), None)
if pop_col is None:
    raise KeyError(
        f"could not identify population total column.\nAvailable: {cols}\n"
        f"searched for: {POP_CANDIDATES}"
    )

print(" column mapping confirmed:")
print(f"LAD code column  : '{lad_col}'")
print(f"Year column      : '{year_col}'")
print(f"Population column: '{pop_col}'")
print(f"Years present in file: {sorted(df_pop_raw[year_col].dropna().unique().tolist())}")

 column mapping confirmed:
LAD code column  : 'geography code'
Year column      : 'year'
Population column: 'All Ages'
Years present in file: [2023, 2024, 2025]


In [23]:
# filter to relevant LADs and map to PFA 
total_rows = len(df_pop_raw)

df_pop_filtered = df_pop_raw[df_pop_raw[lad_col].isin(LAD_TO_PFA)].copy()
df_pop_filtered["force_name"] = df_pop_filtered[lad_col].map(LAD_TO_PFA)

matched_rows = len(df_pop_filtered)
print(f"LAD filter:")
print(f"Total rows in file  : {total_rows:,}")
print(f"Rows matched to PFAs: {matched_rows:,}")
print(f"Rows excluded       : {total_rows - matched_rows:,}")

matched_lads   = df_pop_filtered[lad_col].nunique()
expected_lads  = len(LAD_TO_PFA)
print(f"Distinct LADs matched: {matched_lads} / {expected_lads} expected")

missing_lads = set(LAD_TO_PFA.keys()) - set(df_pop_filtered[lad_col].unique())
if missing_lads:
    print(f"\n {len(missing_lads)} LAD codes expected but not found in file:")
    for code in sorted(missing_lads):
        print(f"{code}  -  {LAD_TO_PFA[code]}")
else:
    print(f"All {expected_lads} expected LAD codes found in file")

# aggregate LAD populations to PFA × Year
# Coerce to numeric (strips commas or whitespace if present in raw file)
df_pop_filtered["pop_numeric"] = (
    df_pop_filtered[pop_col]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)

df_pop_agg = (
    df_pop_filtered
    .groupby(["force_name", year_col], as_index=False)["pop_numeric"]
    .sum()
    .rename(columns={year_col: "year", "pop_numeric": "force_population"})
)

# enforce schema dtypes
df_pop_agg["year"]             = df_pop_agg["year"].astype(int)
df_pop_agg["force_population"] = df_pop_agg["force_population"].astype(int)

print("\n PFA × Year aggregation:")
print(df_pop_agg.sort_values(["force_name", "year"]).to_string(index=False))

LAD filter:
Total rows in file  : 174
Rows matched to PFAs: 174
Rows excluded       : 0
Distinct LADs matched: 58 / 58 expected
All 58 expected LAD codes found in file

 PFA × Year aggregation:
                 force_name  year  force_population
    Devon & Cornwall Police  2023           2682474
    Devon & Cornwall Police  2024           2682474
    Devon & Cornwall Police  2025           2682474
Metropolitan Police Service  2023           9089736
Metropolitan Police Service  2024           9089736
Metropolitan Police Service  2025           9089736
       West Midlands Police  2023           3036605
       West Midlands Police  2024           3036605
       West Midlands Police  2025           3036605
      West Yorkshire Police  2023           2435236
      West Yorkshire Police  2024           2435236
      West Yorkshire Police  2025           2435236


In [24]:
# fill missing years (including 2026
# ONS mid-year estimates cover 2023 and 2024 (2025 may be provisional).
# Crime data extends to March 2026, so we carry forward the most recent available estimate for each force.

# Assumption: 2026 population = most recent ONS mid-year figure (2025 if present, else 2024). This introduces minimal error over a 3-month window

TARGET_YEARS = [2023, 2024, 2025, 2026]
forces   = sorted(df_pop_agg["force_name"].unique())
rows_to_add  = []

for force in forces:
    force_df = df_pop_agg[df_pop_agg["force_name"] == force].set_index("year")
    max_year = force_df.index.max()
    for yr in TARGET_YEARS:
        if yr not in force_df.index:
            pop_val = int(force_df.loc[max_year, "force_population"])
            rows_to_add.append({"force_name": force, "year": yr, "force_population": pop_val})
            print(f"  {force} | {yr}: not in ONS file — "
                  f"carrying forward {max_year} estimate ({pop_val:,})")

if rows_to_add:
    df_pop_agg = pd.concat(
        [df_pop_agg, pd.DataFrame(rows_to_add)], ignore_index=True
    )

# final sort and dtype enforcement
df_pop_agg = df_pop_agg.sort_values(["force_name", "year"]).reset_index(drop=True)
df_pop_agg["year"]             = df_pop_agg["year"].astype(int)
df_pop_agg["force_population"] = df_pop_agg["force_population"].astype(int)

print(f"\n Final population lookup — {len(df_pop_agg)} rows "
      f"({len(forces)} forces x {len(TARGET_YEARS)} years):")
print()
display(df_pop_agg)

  Devon & Cornwall Police | 2026: not in ONS file — carrying forward 2025 estimate (2,682,474)
  Metropolitan Police Service | 2026: not in ONS file — carrying forward 2025 estimate (9,089,736)
  West Midlands Police | 2026: not in ONS file — carrying forward 2025 estimate (3,036,605)
  West Yorkshire Police | 2026: not in ONS file — carrying forward 2025 estimate (2,435,236)

 Final population lookup — 16 rows (4 forces x 4 years):



,force_name,year,force_population
0,Devon & Cornwall Police,2023,2682474
1,Devon & Cornwall Police,2024,2682474
2,Devon & Cornwall Police,2025,2682474
3,Devon & Cornwall Police,2026,2682474
4,Metropolitan Police Service,2023,9089736
5,Metropolitan Police Service,2024,9089736
6,Metropolitan Police Service,2025,9089736
7,Metropolitan Police Service,2026,9089736
8,West Midlands Police,2023,3036605
9,West Midlands Police,2024,3036605


In [25]:
# validate population lookup 
print("Population table validation")

# all 4 forces must be present
missing_forces = set(FORCES) - set(df_pop_agg["force_name"].unique())
if missing_forces:
    raise ValueError(f" Forces missing from population table: {missing_forces}")
print(f"All {len(FORCES)} forces present in population table")

# no null population values
null_count = df_pop_agg["force_population"].isna().sum()
if null_count > 0:
    raise ValueError(f"{null_count} null values in force_population column")
print(f"no null values in force_population")

# Plausibility check — every force must have > 500,000 population
MIN_POP = 500_000
low_pop = df_pop_agg[df_pop_agg["force_population"] < MIN_POP]
if not low_pop.empty:
    print(f"{len(low_pop)} rows with force_population < {MIN_POP:,}:")
    display(low_pop)
else:
    print(f"All force_population values ≥ {MIN_POP:,} (plausibility check passed)")

# join population onto crime DataFrame
rows_before = len(df)

df = df.merge(
    df_pop_agg[["force_name", "year", "force_population"]],
    on=["force_name", "year"],
    how="left",  # left join — no rows dropped
)

rows_after = len(df)

print("Join validation")

# row count must be unchanged (left join)
if rows_before != rows_after:
    raise ValueError(
        f"row count changed after join: {rows_before:,} -> {rows_after:,}. "
        "Possible many-to-many join - investigate."
    )
print(f"Row count preserved: {rows_before:,} -> {rows_after:,}")

# No nulls in force_population after join
pop_nulls = df["force_population"].isna().sum()
if pop_nulls > 0:
    problem_keys = (
        df[df["force_population"].isna()][["force_name", "year"]]
        .drop_duplicates()
    )
    print(f"{pop_nulls:,} records with null force_population after join:")
    display(problem_keys)
else:
    print(f"No null force_population values after join")

# Confirm assigned values per force
print("\n Assigned force_population values:")
display(
    df[["force_name", "year", "force_population"]]
    .drop_duplicates()
    .sort_values(["force_name", "year"])
)


Population table validation
All 4 forces present in population table
no null values in force_population
All force_population values ≥ 500,000 (plausibility check passed)
Join validation
Row count preserved: 5,728,079 -> 5,728,079
No null force_population values after join

 Assigned force_population values:


,force_name,year,force_population
145073,Devon & Cornwall Police,2023,2682474
1583640,Devon & Cornwall Police,2024,2682474
3515289,Devon & Cornwall Police,2025,2682474
5421970,Devon & Cornwall Police,2026,2682474
0,Metropolitan Police Service,2023,9089736
1443234,Metropolitan Police Service,2024,9089736
3379504,Metropolitan Police Service,2025,9089736
5289057,Metropolitan Police Service,2026,9089736
84943,West Midlands Police,2023,3036605
1532330,West Midlands Police,2024,3036605


---
## Section 5 - Aggregation for Reporting

Aggragation to the reporting grain: **Police Force x Month x Crime Category**.

This is the step where the dataset changes from individual crime records to the summarised, BI-ready format. After aggregation, each row represents one unique combination of force, month, and crime type - with all the derived metrics attached.

Key design decisions:
- `crime_count` is a simple row count at grain level.
- `avg_imd_decile` excludes NaN values from the mean - groups where *all* records had null LSOAs will produce NaN, which is documented and expected.
- `force_population` is a constant per force-year, so `first()` correctly carries it through aggregation.
- `crimes_per_1000` is computed *after* aggregation, from the aggregated counts.

In [26]:
# aggregating to reporting grain
GRAIN = ["force_name", "month_period", "crime_type"]

df_agg = (
    df.groupby(GRAIN, observed=True)
    .agg(
        crime_count      = ("Crime ID", "count"),          # COUNT of records (null-safe)
        avg_imd_decile   = ("imd_decile", "mean"),         # mean decile; NaN where all null
        force_population = ("force_population", "first"),  # constant per force-year
        year             = ("year", "first"),
        month_num        = ("month_num", "first"),
        quarter          = ("quarter", "first"),
    )
    .reset_index()
)

print(f"Aggregated dataset shape: {df_agg.shape}")
print(f"Rows at grain: {len(df_agg):,}")

Aggregated dataset shape: (2016, 9)
Rows at grain: 2,016


In [27]:
# preview aggregated data

print(f"Aggregated shape: {df_agg.shape}")
print(f"Columns: {df_agg.columns.tolist()}")
display(df_agg.head())

Aggregated shape: (2016, 9)
Columns: ['force_name', 'month_period', 'crime_type', 'crime_count', 'avg_imd_decile', 'force_population', 'year', 'month_num', 'quarter']


,force_name,month_period,crime_type,crime_count,avg_imd_decile,force_population,year,month_num,quarter
0,Devon & Cornwall Police,2023-04,Anti-Social Behaviour,0,3.750412,2682474,2023,4,2023-Q2
1,Devon & Cornwall Police,2023-04,Bicycle Theft,25,4.473684,2682474,2023,4,2023-Q2
2,Devon & Cornwall Police,2023-04,Burglary,141,4.641221,2682474,2023,4,2023-Q2
3,Devon & Cornwall Police,2023-04,Criminal Damage And Arson,613,3.980144,2682474,2023,4,2023-Q2
4,Devon & Cornwall Police,2023-04,Drugs,192,4.051613,2682474,2023,4,2023-Q2


In [ ]:
# post-aggregation validation 

# dhecking for duplicate rows at the grain
dupes_at_grain = df_agg.duplicated(subset=GRAIN).sum()
assert dupes_at_grain == 0, f"FAIL: {dupes_at_grain} duplicate rows found at the reporting grain!"
print(f"no duplicate rows at grain (force x month x crime_type).")

# checking required fields for nulls
required_fields = ["force_name", "month_period", "crime_type", "crime_count"]
for col in required_fields:
    nulls = df_agg[col].isna().sum()
    status = "check" if nulls == 0 else "fail"
    print(f"  {status}  '{col}' null count: {nulls}")

# summary stats
print()
print(f"Total rows: {len(df_agg):,}")
print(f"Unique forces: {df_agg['force_name'].nunique()} -> {df_agg['force_name'].unique().tolist()}")
print(f"Unique months: {df_agg['month_period'].nunique()}")
print(f"Unique crime types: {df_agg['crime_type'].nunique()}")
print(f"crime_count range: {df_agg['crime_count'].min()} - {df_agg['crime_count'].max():,}")

no duplicate rows at grain (force x month x crime_type).
  check  'force_name' null count: 0
  check  'month_period' null count: 0
  check  'crime_type' null count: 0
  check  'crime_count' null count: 0

Total rows: 2,016
Unique forces: 4 -> ['Devon & Cornwall Police', 'Metropolitan Police Service', 'West Midlands Police', 'West Yorkshire Police']
Unique months: 36
Unique crime types: 14
crime_count range: 0 – 25,800


---
## Section 6 — Severity Score Join 

**Why this dataset?**  
The Home Office Crime Severity Score weights crimes by their seriousness (based on sentencing guidelines), rather than treating all crimes equally. 

**Source:** Home Office Crime Severity Score Data Tool (`.xls` format, processed with `xlrd`).

**Known limitation:** The most recent available year in this file is **April 2015 – March 2016**. I'm using this as a static baseline proxy for the 2023–2026 period. Crime severity weights do evolve over time (as sentencing guidelines change), so this figure should be treated as an indicative baseline, not a current measurement. This assumption is clearly documented.

**Grain of join:** Force-level only. The severity score is a single value per force, joined as a static lookup onto the aggregated dataset.

In [30]:
# ensure xlrd is available 
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "xlrd", "-q"])

# load severity score XLS
import xlrd

wb = xlrd.open_workbook(SEVERITY_FILE)
sh = wb.sheet_by_name("Data")

# extract header row 
headers = [sh.cell_value(1, j) for j in range(sh.ncols)]

# build a list of dicts for all rows of data
rows_data = []
for i in range(2, sh.nrows):
    row_dict = {headers[j]: sh.cell_value(i, j) for j in range(sh.ncols)}
    rows_data.append(row_dict)

df_severity_raw = pd.DataFrame(rows_data)

print(f"severity data loaded: {df_severity_raw.shape}")
print(f"last column (most recent period): '{headers[-1]}'")
print()
# show which forces are in the file
print("force names in severity file:")
print(df_severity_raw["Name"].tolist())

severity data loaded: (55, 16)
last column (most recent period): 'Apr '15 to Mar '16'

force names in severity file:
['ENGLAND AND WALES', 'ENGLAND', 'North East', 'Cleveland', 'Durham', 'Northumbria', 'North West', 'Cheshire', 'Cumbria', 'Greater Manchester', 'Lancashire', 'Merseyside', 'Yorkshire and The Humber', 'Humberside', 'North Yorkshire', 'South Yorkshire', 'West Yorkshire', 'East Midlands', 'Derbyshire', 'Leicestershire', 'Lincolnshire', 'Northamptonshire', 'Nottinghamshire', 'West Midlands', 'Staffordshire', 'Warwickshire', 'West Mercia', 'West Midlands', 'East', 'Bedfordshire', 'Cambridgeshire', 'Essex', 'Hertfordshire', 'Norfolk', 'Suffolk', 'London', 'City of London', 'Metropolitan Police', 'South East', 'Hampshire', 'Kent', 'Surrey', 'Sussex', 'Thames Valley', 'South West', 'Avon and Somerset', 'Devon and Cornwall', 'Dorset', 'Gloucestershire', 'Wiltshire', 'WALES', 'Dyfed-Powys', 'Gwent', 'North Wales', 'South Wales']


In [31]:
# extract the four target forces and most recent severity scores
SEVERITY_COL = "Apr '15 to Mar '16"  # most recent available year

sev_rows = []
for raw_name, canonical_name in SEVERITY_NAME_MAP.items():
    match = df_severity_raw[df_severity_raw["Name"] == raw_name]
    if match.empty:
        print(f" '{raw_name}' not found in severity file - check SEVERITY_NAME_MAP.")
        continue
    score = match.iloc[0][SEVERITY_COL]
    sev_rows.append({"force_name": canonical_name, "severity_score_baseline": float(score)})

df_severity = pd.DataFrame(sev_rows)
print("severity score lookup table:")
display(df_severity)

print()
print(f"assumption: '{SEVERITY_COL}' score used as a static proxy baseline for 2023-2026.")
print("this is the most recent year available in the Home Office severity data tool.")


severity score lookup table:


,force_name,severity_score_baseline
0,Metropolitan Police Service,86.570643
1,West Midlands Police,62.651533
2,West Yorkshire Police,89.749767
3,Devon & Cornwall Police,45.181490



assumption: 'Apr '15 to Mar '16' score used as a static proxy baseline for 2023-2026.
this is the most recent year available in the Home Office severity data tool.


In [32]:
# join severity scores onto aggregated dataset
df_final = df_agg.merge(
    df_severity,
    on="force_name",
    how="left"
)

sev_nulls = df_final["severity_score_baseline"].isna().sum()
if sev_nulls > 0:
    print(f" {sev_nulls} rows have no severity score — check name mapping.")
else:
    print(" Severity score joined successfully for all forces.")
    display(df_final[["force_name", "severity_score_baseline"]].drop_duplicates())

# compute crimes_per_1000 on the final DataFrame
df_final["crimes_per_1000"] = (
    df_final["crime_count"] / df_final["force_population"]
) * 1000
df_final["crimes_per_1000"] = df_final["crimes_per_1000"].round(4)

print("\n  crimes_per_1000 computed.")
display(df_final[["force_name", "crime_count", "force_population", "crimes_per_1000"]].head())

 Severity score joined successfully for all forces.


,force_name,severity_score_baseline
0,Devon & Cornwall Police,45.181490
504,Metropolitan Police Service,86.570643
1008,West Midlands Police,62.651533
1512,West Yorkshire Police,89.749767



  crimes_per_1000 computed.


,force_name,crime_count,force_population,crimes_per_1000
0,Devon & Cornwall Police,0,2682474,0.0000
1,Devon & Cornwall Police,25,2682474,0.0093
2,Devon & Cornwall Police,141,2682474,0.0526
3,Devon & Cornwall Police,613,2682474,0.2285
4,Devon & Cornwall Police,192,2682474,0.0716


---
## Section 7 - Export Layer

The pipeline is complete. Data will be exported into the final reporting dataset to `outputs/crime_reporting_dataset.csv`, then run a final validation sweep to confirm everything is in order.

In [33]:
# column ordering for BI readability 
FINAL_COLUMNS = [
    "force_name",
    "month_period",
    "crime_type",
    "year",
    "month_num",
    "quarter",
    "crime_count",
    "force_population",
    "crimes_per_1000",
    "avg_imd_decile",
    "severity_score_baseline",
]

df_output = df_final[FINAL_COLUMNS].copy()
df_output = df_output.sort_values(["force_name", "month_period", "crime_type"]).reset_index(drop=True)

print(f"final columns  : {df_output.columns.tolist()}")
print(f"final shape    : {df_output.shape}")

final columns  : ['force_name', 'month_period', 'crime_type', 'year', 'month_num', 'quarter', 'crime_count', 'force_population', 'crimes_per_1000', 'avg_imd_decile', 'severity_score_baseline']
final shape    : (2016, 11)


In [34]:
# export to CSV 
df_output.to_csv(OUTPUT_FILE, index=False)
print(f"exported to: {OUTPUT_FILE}")
print(f"total file size: {os.path.getsize(OUTPUT_FILE) / 1e3:.1f} KB")

exported to: c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\outputs\crime_reporting_dataset.csv
total file size: 239.1 KB


In [35]:
# validation
print("FINAL VALIDATION")
print(f"total records: {len(df_output):,}")
print(f"date range: {df_output['month_period'].min()} -> {df_output['month_period'].max()}")
print(f"distinct forces: {df_output['force_name'].nunique()}")
print(f"distinct crime types: {df_output['crime_type'].nunique()}")
print(f"Min crime_count: {df_output['crime_count'].min()}")
print(f"Max crime_count: {df_output['crime_count'].max():,}")
print()
print("  Null counts per column:")
null_summary = df_output.isnull().sum()
for col, n in null_summary.items():
    flag = "error" if n > 0 else "pass"
    print(f"    {flag}  {col:<30} {n:>6,}")
print()
print("data types:")
for col, dtype in df_output.dtypes.items():
    print(f"    {col:<30} {str(dtype)}")
print()
print("sample output (5 rows):")
display(df_output.sample(5, random_state=42))


FINAL VALIDATION
total records: 2,016
date range: 2023-04 -> 2026-03
distinct forces: 4
distinct crime types: 14
Min crime_count: 0
Max crime_count: 25,800

  Null counts per column:
    pass  force_name                          0
    pass  month_period                        0
    pass  crime_type                          0
    pass  year                                0
    pass  month_num                           0
    pass  quarter                             0
    pass  crime_count                         0
    pass  force_population                    0
    pass  crimes_per_1000                     0
    pass  avg_imd_decile                      0
    pass  severity_score_baseline             0

data types:
    force_name                     object
    month_period                   object
    crime_type                     object
    year                           int32
    month_num                      int32
    quarter                        object
    crime_count           

,force_name,month_period,crime_type,year,month_num,quarter,crime_count,force_population,crimes_per_1000,avg_imd_decile,severity_score_baseline
1198,West Midlands Police,2024-05,Public Order,2024,5,2024-Q2,1929,3036605,0.6352,2.886971,62.651533
526,Metropolitan Police Service,2023-05,Public Order,2023,5,2023-Q2,4982,9089736,0.5481,4.384699,86.570643
393,Devon & Cornwall Police,2025-08,Bicycle Theft,2025,8,2025-Q3,61,2682474,0.0227,4.500000,45.181490
1407,West Midlands Police,2025-08,Possession Of Weapons,2025,8,2025-Q3,584,3036605,0.1923,2.346743,62.651533
433,Devon & Cornwall Police,2025-10,Violence And Sexual Offences,2025,10,2025-Q4,5546,2682474,2.0675,4.195016,45.181490


---
## Section 8 - Data Dictionary

This data dictionary documents every column in the final output file, including its type, source, derivation logic, and any assumptions made. It is intended to serve as the handoff document for the Power BI analytics team.

**Reporting grain:** One row per unique combination of `force_name` × `month_period` × `crime_type`.

| Column | Data Type | Description | Source | Assumption |
|---|---|---|---|---|
| `force_name` | str | Canonical police force name | Police.uk (cleaned) | Normalised from raw `Falls within` column using a force name mapping dictionary |
| `month_period` | str | Reporting month in `YYYY-MM` format | Police.uk | Preserved from raw `Month` field |
| `crime_type` | str | Standardised crime category label | Police.uk | Stripped of whitespace and title-cased to ensure consistency |
| `year` | int | Calendar year extracted from month | Derived | `month_dt.year` - used to join population and for annual filtering |
| `month_num` | int | Month number 1-12 | Derived | `month_dt.month` - supports seasonality analysis in Power BI |
| `quarter` | str | Calendar quarter label in YYYY-QN format | Derived | Concatenation of `year` and `ceil(month_num / 3)` |
| `crime_count` | int | Total crime records at this grain | Police.uk | COUNT of rows per force x month x crime_type group |
| `force_population` | int | Estimated resident population of the force area | ONS mid-year estimates + hardcoded | Annual ONS-aligned estimate |
| `crimes_per_1000` | float | Crimes per 1,000 residents | Derived | `(crime_count / force_population) × 1000` - enables fair cross-force comparison |
| `avg_imd_decile` | float | Mean IMD deprivation decile for crimes in this group | IMD 2019 (MHCLG) | Mean of LSOA-level IMD decile values at the grain; NaN where all records had null LSOAs. Decile 1 = most deprived; Decile 10 = least deprived. |
| `severity_score_baseline` | float | Crime Severity Score for the force | Home Office Crime Severity Score Data Tool | Apr 2015- Mar 2016. Used as a static force-level proxy. Severity weights reflect sentencing guidelines and may have changed since 2016 |

ai disclosure: ai was utilised to format markdown